# Create Fact Table

Reading silver data

In [0]:
df_car_sales = (
    spark.read.format('parquet')
    .option('header','true')
    .load('abfss://automotive-sales@automotivesalessa.dfs.core.windows.net/silver/car_sales')
)

Reading all dims

In [0]:
df_dim_dealer = spark.sql("select * from cars_catalog.gold.dim_dealer")
df_dim_branch = spark.sql("select * from cars_catalog.gold.dim_branch")
df_dim_model = spark.sql("select * from cars_catalog.gold.dim_model")
df_dim_date = spark.sql("select * from cars_catalog.gold.dim_date")

### Bringing keys to fact table

In [0]:
df_fact = (
    df_car_sales.join(df_dim_dealer, "Dealer_ID", "left")
               .join(df_dim_branch, "Branch_ID", "left")
               .join(df_dim_model, "Model_ID", "left")
               .join(df_dim_date, "Date_ID", "left")
               .select(
                   df_car_sales["Revenue"],
                   df_car_sales["Units_Sold"],
                   df_car_sales["Per_Unit_Revenue"],
                   df_dim_dealer["dim_dealer_key"],
                   df_dim_branch["dim_branch_key"],
                   df_dim_model["dim_model_key"],
                   df_dim_date["dim_date_key"]
               )
)

Write the fact table

In [0]:
from delta.tables import DeltaTable

In [0]:
(
    df_fact.write
    .format('delta')
    .mode('overwrite')
    .option('path','abfss://automotive-sales@automotivesalessa.dfs.core.windows.net/gold/fact_sales')
    .saveAsTable('cars_catalog.gold.fact_sales')
)

In [0]:
%sql
select * from cars_catalog.gold.fact_sales